Title: LEE_Detection_ERA5_1960_.ipynb

Purpose: Identifiy Low Energy Events with varying lengths from the model output data for all years of ERA5 data

Author: Onno Nennecke on 21.03.2025 Modified: 07.04.2026

Input data: 

- adjusted final model output
    - This file lies here: '/climca/people/onennecke/model_output/not_bias_corrected/full_year/ERA5_all_years/ERA5_hist_timeseries.nc'

Output data:

- LEE Tables: LEE_dat_14.csv, LEE_dat_7.csv, LEE_dat.csv, LEE_dat_14_selection.csv, LEE_dat_7_selection.csv, LEE_vl.csv
    - This file lies here: '/climca/people/onennecke/model_output/LEE_detection/'

In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
import os
import glob


### Read Model output data

In [2]:
# path = '/climca/people/onennecke/model_output/not_bias_corrected/model_output_adj.nc'
path = '/climca/people/onennecke/model_output/not_bias_corrected/model_output_all.nc'
# path = '/climca/people/onennecke/model_output/not_bias_corrected/model_output_all_future.nc'
path = '/climca/people/onennecke/model_output/not_bias_corrected/full_year/ERA5_all_years/ERA5_hist_timeseries.nc'

ts_datasets = xr.open_dataset(path)
ts_datasets.load()

<xarray.Dataset> Size: 2MB
Dimensions:        (time: 23725)
Coordinates:
  * time           (time) datetime64[ns] 190kB 1960-01-01 ... 2024-12-31
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    ESM            <U9 36B 'ERA5_week'
    run            <U4 16B 'hist'
    ESM_run        <U14 56B 'ERA5_hist_week'
    country        float64 8B 9.0
    period         <U4 16B 'week'
Data variables:
    temp           (time) float64 190kB 7.185 7.429 6.072 ... 0.7396 0.4142
    demand         (time) float64 190kB 1.43e+03 1.427e+03 ... 1.515e+03
    sfcWind        (time) float32 95kB 8.02 5.214 3.839 ... 6.159 8.049 8.775
    rsds           (time) float32 95kB 24.19 16.84 14.44 ... 28.84 26.3 31.92
    tas            (time) float32 95kB 6.337 6.546 5.314 ... 0.2238 1.512 1.194
    tasmax         (time) float32 95kB 7.993 7.899 6.449 ... 2.446 2.709 3.361
    wind_off_prod  (time) float64 190kB 167.1 8.772 4.656 ... 179.2 221.1 221.1
    wind_on_prod   (time) float64 190kB 401.3 84.65 27.66 ... 216.6 592.8 602.7
    solar_prod     (time) float64 190kB 63.45 47.02 31.76 ... 78.49 72.3 91.06
    total_prod     (time) float64 190kB 631.9 140.4 64.07 ... 474.3 886.3 914.8
    Netto          (time) float64 190kB -798.1 -1.286e+03 ... -624.5 -600.0
    Residual_load  (time) float64 190kB 798.1 1.286e+03 1.38e+03 ... 624.5 600.0

In [3]:
# path = '/climca/people/onennecke/model_output/bias_corrected_masked_ibicus/full_year/'
# files = sorted(glob.glob(path + '*.nc'))
# # files = files[:60] + files[61:]
# ts_datasets = xr.open_mfdataset(files, combine='nested', concat_dim = 'ESM_run')
# ts_datasets.coords['doy'] = (('time',), np.tile(np.arange(1, 366), 10))

# ts_datasets

#### Identification of low energy events

In [4]:
# RL = ts_datasets['Residual_load_adj']
RL = ts_datasets['Residual_load']

RL.values

array([ 798.09080076, 1286.41945389, 1380.19571113, ..., 1053.88900169,
        624.51341296,  599.97315407])

In [5]:
# Calculate the rolling means
ts_datasets['RL_mov_avg_7'] = RL_mov_avg_7 = RL.rolling(time=7, center=False).mean()
ts_datasets['RL_mov_avg_14'] = RL_mov_avg_14 = RL.rolling(time=14, center=False).mean()
# ts_datasets

In [6]:
ts_datasets.load()

<xarray.Dataset> Size: 2MB
Dimensions:        (time: 23725)
Coordinates:
  * time           (time) datetime64[ns] 190kB 1960-01-01 ... 2024-12-31
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    ESM            <U9 36B 'ERA5_week'
    run            <U4 16B 'hist'
    ESM_run        <U14 56B 'ERA5_hist_week'
    country        float64 8B 9.0
    period         <U4 16B 'week'
Data variables: (12/14)
    temp           (time) float64 190kB 7.185 7.429 6.072 ... 0.7396 0.4142
    demand         (time) float64 190kB 1.43e+03 1.427e+03 ... 1.515e+03
    sfcWind        (time) float32 95kB 8.02 5.214 3.839 ... 6.159 8.049 8.775
    rsds           (time) float32 95kB 24.19 16.84 14.44 ... 28.84 26.3 31.92
    tas            (time) float32 95kB 6.337 6.546 5.314 ... 0.2238 1.512 1.194
    tasmax         (time) float32 95kB 7.993 7.899 6.449 ... 2.446 2.709 3.361
    ...             ...
    solar_prod     (time) float64 190kB 63.45 47.02 31.76 ... 78.49 72.3 91.06
    total_prod     (time) float64 190kB 631.9 140.4 64.07 ... 474.3 886.3 914.8
    Netto          (time) float64 190kB -798.1 -1.286e+03 ... -624.5 -600.0
    Residual_load  (time) float64 190kB 798.1 1.286e+03 1.38e+03 ... 624.5 600.0
    RL_mov_avg_7   (time) float64 190kB nan nan nan ... 1.215e+03 1.11e+03
    RL_mov_avg_14  (time) float64 190kB nan nan nan nan ... 900.7 938.9 923.8

In [7]:
print(ts_datasets.Residual_load.values[0:10])
print(ts_datasets.RL_mov_avg_7.values[0:10])

[ 798.09080076 1286.41945389 1380.19571113 1330.88743896 1068.23496038
  557.6102914   905.96333628 1194.71860866  899.39381807 1401.09995761]
[          nan           nan           nan           nan           nan
           nan 1046.77171326 1103.43282867 1048.14345213 1051.12977305]


##### Threshold Calculation

In [8]:
thresh_perc = 0.95
threshold_week = np.float64(1319.138650687668)
threshold_week

np.float64(1319.138650687668)

In [9]:
# Time series of "True" when threshold is exceeded, "False" otherwise

exceed_bool_1 = xr.where(RL > threshold_week, True, False)
exceed_bool_7 = xr.where(RL_mov_avg_7 > threshold_week, True, False)
exceed_bool_14 = xr.where(RL_mov_avg_14 > threshold_week, True, False)

In [10]:
# Look for events without any rolling mean


# Count number of true values overall
count_exceed_1 = exceed_bool_1.sum(dim='time')
count_exceed_1.values
# exceed_bool_1.time
# np.zeros_like(exceed_bool_1, dtype=int)

array(1568)

### Days above threshold (dat) 

- Take each day as its own event (not really events but just days above threshold)

In [11]:
# 1) Extract the flat indices where mask is True
time_idx = np.nonzero(exceed_bool_1.values)
run_idx = np.repeat(0, len(time_idx[0]))
n_dat = run_idx.size

# 2) Create a flat counter 1…n_dat
labels = np.arange(1, n_dat + 1, dtype=int)

# 3) Scatter them back into an integer array of same shape
dat = np.zeros_like(exceed_bool_1.values, dtype=int)
dat[time_idx] = labels

# wrap back into an xarray
dat = xr.DataArray(
    dat,
    coords=exceed_bool_1.coords,
    dims=exceed_bool_1.dims,
    name="dat"
)

n_dat = n_dat
dat

<xarray.DataArray 'dat' (time: 23725)> Size: 190kB
array([0, 0, 1, ..., 0, 0, 0])
Coordinates:
  * time      (time) datetime64[ns] 190kB 1960-01-01 1960-01-02 ... 2024-12-31
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
    ESM       <U9 36B 'ERA5_week'
    run       <U4 16B 'hist'
    ESM_run   <U14 56B 'ERA5_hist_week'
    country   float64 8B 9.0
    period    <U4 16B 'week'

In [12]:
# 1) Extract the flat indices where mask is True
time_idx = np.nonzero(exceed_bool_7.values)
run_idx = np.repeat(0, len(time_idx[0]))

n_dat_7 = run_idx.size

# 2) Create a flat counter 1…n_dat_7
labels = np.arange(1, n_dat_7 + 1, dtype=int)

# 3) Scatter them back into an integer array of same shape
dat_7 = np.zeros_like(exceed_bool_7.values, dtype=int)
dat_7[time_idx] = labels

# wrap back into an xarray
dat_7 = xr.DataArray(
    dat_7,
    coords=exceed_bool_7.coords,
    dims=exceed_bool_7.dims,
    name="dat_7"
)

n_dat_7 = n_dat_7
dat_7

<xarray.DataArray 'dat_7' (time: 23725)> Size: 190kB
array([0, 0, 0, ..., 0, 0, 0])
Coordinates:
  * time      (time) datetime64[ns] 190kB 1960-01-01 1960-01-02 ... 2024-12-31
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
    ESM       <U9 36B 'ERA5_week'
    run       <U4 16B 'hist'
    ESM_run   <U14 56B 'ERA5_hist_week'
    country   float64 8B 9.0
    period    <U4 16B 'week'

In [13]:
# 1) Extract the flat indices where mask is True
time_idx = np.nonzero(exceed_bool_14.values)
run_idx = np.repeat(0, len(time_idx[0]))

n_dat_14 = run_idx.size

# 2) Create a flat counter 1…n_dat_14
labels = np.arange(1, n_dat_14 + 1, dtype=int)

# 3) Scatter them back into an integer array of same shape
dat_14 = np.zeros_like(exceed_bool_14.values, dtype=int)
dat_14[time_idx] = labels

# wrap back into an xarray
dat_14 = xr.DataArray(
    dat_14,
    coords=exceed_bool_14.coords,
    dims=exceed_bool_14.dims,
    name="dat_14"
)

n_dat_14 = n_dat_14
dat_14

<xarray.DataArray 'dat_14' (time: 23725)> Size: 190kB
array([0, 0, 0, ..., 0, 0, 0])
Coordinates:
  * time      (time) datetime64[ns] 190kB 1960-01-01 1960-01-02 ... 2024-12-31
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
    ESM       <U9 36B 'ERA5_week'
    run       <U4 16B 'hist'
    ESM_run   <U14 56B 'ERA5_hist_week'
    country   float64 8B 9.0
    period    <U4 16B 'week'

### Events with rolling mean of 1 above threshold (events_vl) = events with varying length 

In [14]:
# Look for events without any rolling mean

events_vl = np.zeros_like(exceed_bool_1, dtype=int)


labeled_segment, num_features = ndimage.label(exceed_bool_1.values)
current_label = 1

if num_features > 0:
    labeled_segment[labeled_segment > 0] += current_label - 1
    current_label += num_features
events_vl = labeled_segment

# # Starting value for the labels
# current_label = 1
# counter = 0
# for run in exceed_bool_1.ESM_run.values:
#     run_data = exceed_bool_1.sel(ESM_run=run)
#     # Only label this section
#     labeled_segment, num_features = ndimage.label(run_data.values)

#     if num_features > 0:
#         labeled_segment[labeled_segment > 0] += current_label - 1
#         current_label += num_features

#     # Save the label to the result array
#     events_vl[counter] = labeled_segment
#     counter += 1

n_events_vl = current_label - 1
n_events_vl



779

In [59]:
# def LEE_detection(events, n_events, t=RL['time'].values, dur = 7, minDuration=1):
    
# events = dat
# n_events = n_dat
# t=RL['time'].values
# dur = 1
# minDuration=1




def LEE_detection(events, n_events, t=RL['time'].values, dur = 7, minDuration=1):
    LEE_records = []
    for ev in range(1, n_events + 1):
        event_duration = (events == ev).sum()
        if event_duration < minDuration:
            continue

        end_idx = np.where(events == ev)[0][0]
        start_idx = end_idx - dur + 1

        date_start = t[start_idx]
        date_end = t[end_idx]

        LEE_start = np.where(t == date_start)[0][0]
        LEE_end = np.where(t == date_end)[0][0]

        RL_run = RL.values
        RL_LEE = RL_run[LEE_start:LEE_end + 1]
        LEE_peak = np.argmax(RL_LEE)

        record = {
            'date_start': date_start,
            'date_end': date_end,
            'date_peak': date_start + LEE_peak,
            # 'date_start_old': RL['old_time'][i][LEE_start].values,
            # 'date_end_old': RL['old_time'][i][LEE_end].values,
            # 'date_peak_old': RL['old_time'][i][LEE_start + LEE_peak].values,
            'index_start': LEE_start,
            'index_end': LEE_end,
            'index_peak': LEE_start + LEE_peak,
            'duration': len(RL_LEE),
            'RL_max': RL_LEE[LEE_peak],
            'RL_mean': RL_LEE.mean(),
            'RL_var': np.sqrt(RL_LEE.var()),
            'RL_cumulative': RL_LEE.sum(),
            'event': ev,
            'ESM': str(RL.ESM.values),
            'ESM_run': str(RL.ESM_run.values),
            'year': RL.time.dt.year[start_idx].values,
            # 'doy' : RL.doy[start_idx].values#,
            # 'winter': RL.winter_year[start_idx].values,
            # 'day_of_winter': RL.day_of_winter[start_idx].values
        }

        # # Optional additional metrics if available
        # # (you could generalize this to loop over var names too)
        # for var in ['prod', 'demand', 'pot']:
        #     try:
        #         var_data = RL[var].sel(ESM_run=RL.ESM_run[i]).values[LEE_start:LEE_end + 1]
        #         record[f'{var}_max'] = var_data.max()
        #         record[f'{var}_mean'] = var_data.mean()
        #         record[f'{var}_var'] = np.sqrt(var_data.var())
        #         record[f'{var}_cumulative'] = var_data.sum()
        #     except KeyError:
        #         pass  # Variable doesn't exist, skip

        LEE_records.append(record)

    LEE_dat = pd.DataFrame(LEE_records)
    return LEE_dat


In [18]:
LEE_dat = LEE_detection(dat, n_dat, dur = 1)

In [19]:
LEE_dat

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-03,1960-01-03,1960-01-03,2,2,2,1,1380.195711,1380.195711,0.0,1380.195711,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-04,1960-01-04,1960-01-04,3,3,3,1,1330.887439,1330.887439,0.0,1330.887439,2,ERA5_week,ERA5_hist_week,1960
2,1960-01-10,1960-01-10,1960-01-10,9,9,9,1,1401.099958,1401.099958,0.0,1401.099958,3,ERA5_week,ERA5_hist_week,1960
3,1960-01-12,1960-01-12,1960-01-12,11,11,11,1,1392.550256,1392.550256,0.0,1392.550256,4,ERA5_week,ERA5_hist_week,1960
4,1960-01-15,1960-01-15,1960-01-15,14,14,14,1,1473.575500,1473.575500,0.0,1473.575500,5,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1563,2024-12-13,2024-12-13,2024-12-13,23706,23706,23706,1,1403.400483,1403.400483,0.0,1403.400483,1564,ERA5_week,ERA5_hist_week,2024
1564,2024-12-24,2024-12-24,2024-12-24,23717,23717,23717,1,1337.232681,1337.232681,0.0,1337.232681,1565,ERA5_week,ERA5_hist_week,2024
1565,2024-12-26,2024-12-26,2024-12-26,23719,23719,23719,1,1379.572757,1379.572757,0.0,1379.572757,1566,ERA5_week,ERA5_hist_week,2024
1566,2024-12-27,2024-12-27,2024-12-27,23720,23720,23720,1,1415.401550,1415.401550,0.0,1415.401550,1567,ERA5_week,ERA5_hist_week,2024


In [20]:
LEE_dat.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_ERA5_1960_.csv', index=False)
# LEE_dat.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat.csv', index=False)

LEE_dat

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-03,1960-01-03,1960-01-03,2,2,2,1,1380.195711,1380.195711,0.0,1380.195711,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-04,1960-01-04,1960-01-04,3,3,3,1,1330.887439,1330.887439,0.0,1330.887439,2,ERA5_week,ERA5_hist_week,1960
2,1960-01-10,1960-01-10,1960-01-10,9,9,9,1,1401.099958,1401.099958,0.0,1401.099958,3,ERA5_week,ERA5_hist_week,1960
3,1960-01-12,1960-01-12,1960-01-12,11,11,11,1,1392.550256,1392.550256,0.0,1392.550256,4,ERA5_week,ERA5_hist_week,1960
4,1960-01-15,1960-01-15,1960-01-15,14,14,14,1,1473.575500,1473.575500,0.0,1473.575500,5,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1563,2024-12-13,2024-12-13,2024-12-13,23706,23706,23706,1,1403.400483,1403.400483,0.0,1403.400483,1564,ERA5_week,ERA5_hist_week,2024
1564,2024-12-24,2024-12-24,2024-12-24,23717,23717,23717,1,1337.232681,1337.232681,0.0,1337.232681,1565,ERA5_week,ERA5_hist_week,2024
1565,2024-12-26,2024-12-26,2024-12-26,23719,23719,23719,1,1379.572757,1379.572757,0.0,1379.572757,1566,ERA5_week,ERA5_hist_week,2024
1566,2024-12-27,2024-12-27,2024-12-27,23720,23720,23720,1,1415.401550,1415.401550,0.0,1415.401550,1567,ERA5_week,ERA5_hist_week,2024


In [60]:
LEE_dat_7 = LEE_detection(dat_7, n_dat_7, dur = 7)

In [62]:
# If the date_start is the the next day from the line before and they are the same ESM_run they should get the same event number
new_run = LEE_dat_7['ESM_run'].ne(LEE_dat_7['ESM_run'].shift())
not_consecutive = (LEE_dat_7['date_start'] - LEE_dat_7['date_start'].shift()) != pd.Timedelta(days=1)

# Combine them — whenever either is True, that row is the start of a new event
is_new_event = new_run | not_consecutive

# 4. Cum-sum that to get a 1-based event ID
LEE_dat_7['event'] = is_new_event.cumsum()
LEE_dat_7

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-10,1960-01-16,1960-01-10 00:00:00.000000005,9,15,14,7,1473.575500,1338.173258,131.480939,9367.212803,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-11,1960-01-17,1960-01-11 00:00:00.000000006,10,16,16,7,1481.905678,1349.716932,139.784071,9448.018524,1,ERA5_week,ERA5_hist_week,1960
2,1960-12-11,1960-12-17,1960-12-11 00:00:00.000000002,344,350,346,7,1462.470364,1347.257739,83.400705,9430.804175,2,ERA5_week,ERA5_hist_week,1960
3,1960-12-12,1960-12-18,1960-12-12 00:00:00.000000001,345,351,346,7,1462.470364,1379.305214,69.829424,9655.136500,2,ERA5_week,ERA5_hist_week,1960
4,1960-12-13,1960-12-19,1960-12-13 00:00:00.000000000,346,352,346,7,1462.470364,1376.300830,70.246887,9634.105809,2,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,2023-11-26,2023-12-02,2023-11-26 00:00:00.000000006,23324,23330,23330,7,1482.179304,1381.720272,83.433944,9672.041901,106,ERA5_week,ERA5_hist_week,2023
442,2023-11-27,2023-12-03,2023-11-27 00:00:00.000000005,23325,23331,23330,7,1482.179304,1386.899926,80.895717,9708.299484,106,ERA5_week,ERA5_hist_week,2023
443,2023-11-28,2023-12-04,2023-11-28 00:00:00.000000004,23326,23332,23330,7,1482.179304,1365.415641,116.693246,9557.909487,106,ERA5_week,ERA5_hist_week,2023
444,2023-11-30,2023-12-06,2023-11-30 00:00:00.000000002,23328,23334,23330,7,1482.179304,1324.507752,155.275138,9271.554263,107,ERA5_week,ERA5_hist_week,2023


In [63]:
LEE_dat_7.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_7_ERA5_1960_.csv', index=False)
# LEE_dat_7.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_7.csv', index=False)

LEE_dat_7

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-10,1960-01-16,1960-01-10 00:00:00.000000005,9,15,14,7,1473.575500,1338.173258,131.480939,9367.212803,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-11,1960-01-17,1960-01-11 00:00:00.000000006,10,16,16,7,1481.905678,1349.716932,139.784071,9448.018524,1,ERA5_week,ERA5_hist_week,1960
2,1960-12-11,1960-12-17,1960-12-11 00:00:00.000000002,344,350,346,7,1462.470364,1347.257739,83.400705,9430.804175,2,ERA5_week,ERA5_hist_week,1960
3,1960-12-12,1960-12-18,1960-12-12 00:00:00.000000001,345,351,346,7,1462.470364,1379.305214,69.829424,9655.136500,2,ERA5_week,ERA5_hist_week,1960
4,1960-12-13,1960-12-19,1960-12-13 00:00:00.000000000,346,352,346,7,1462.470364,1376.300830,70.246887,9634.105809,2,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,2023-11-26,2023-12-02,2023-11-26 00:00:00.000000006,23324,23330,23330,7,1482.179304,1381.720272,83.433944,9672.041901,106,ERA5_week,ERA5_hist_week,2023
442,2023-11-27,2023-12-03,2023-11-27 00:00:00.000000005,23325,23331,23330,7,1482.179304,1386.899926,80.895717,9708.299484,106,ERA5_week,ERA5_hist_week,2023
443,2023-11-28,2023-12-04,2023-11-28 00:00:00.000000004,23326,23332,23330,7,1482.179304,1365.415641,116.693246,9557.909487,106,ERA5_week,ERA5_hist_week,2023
444,2023-11-30,2023-12-06,2023-11-30 00:00:00.000000002,23328,23334,23330,7,1482.179304,1324.507752,155.275138,9271.554263,107,ERA5_week,ERA5_hist_week,2023


In [64]:
# find the index of the row with the highest RL_cumulative in each event
idx = LEE_dat_7.groupby('event')['RL_cumulative'].idxmax()

# select only those rows
LEE_dat_7_max_cum_RL = LEE_dat_7.loc[idx].reset_index(drop=True)
LEE_dat_7_max_cum_RL


,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-11,1960-01-17,1960-01-11 00:00:00.000000006,10,16,16,7,1481.905678,1349.716932,139.784071,9448.018524,1,ERA5_week,ERA5_hist_week,1960
1,1960-12-15,1960-12-21,1960-12-15 00:00:00.000000003,348,354,351,7,1443.003131,1394.029694,40.972172,9758.207859,2,ERA5_week,ERA5_hist_week,1960
2,1961-12-22,1961-12-28,1961-12-22 00:00:00.000000004,720,726,724,7,1540.029319,1428.591291,66.012019,10000.139036,3,ERA5_week,ERA5_hist_week,1961
3,1962-11-17,1962-11-23,1962-11-17 00:00:00.000000004,1050,1056,1054,7,1438.360162,1333.234734,97.141039,9332.643138,4,ERA5_week,ERA5_hist_week,1962
4,1962-11-21,1962-11-27,1962-11-21 00:00:00.000000000,1054,1060,1054,7,1438.360162,1354.663811,108.115619,9482.646680,5,ERA5_week,ERA5_hist_week,1962
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,2021-12-20,2021-12-26,2021-12-20 00:00:00.000000001,22618,22624,22619,7,1449.698174,1332.160572,68.921964,9325.124005,103,ERA5_week,ERA5_hist_week,2021
103,2022-12-10,2022-12-16,2022-12-10 00:00:00.000000006,22973,22979,22979,7,1491.687275,1426.349650,29.632589,9984.447552,104,ERA5_week,ERA5_hist_week,2022
104,2023-01-22,2023-01-28,2023-01-22 00:00:00.000000006,23016,23022,23022,7,1387.796806,1351.664366,47.479486,9461.650559,105,ERA5_week,ERA5_hist_week,2023
105,2023-11-27,2023-12-03,2023-11-27 00:00:00.000000005,23325,23331,23330,7,1482.179304,1386.899926,80.895717,9708.299484,106,ERA5_week,ERA5_hist_week,2023


In [65]:
# LEE_dat_7_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_7_selection.csv', index=False)
LEE_dat_7_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_7_selection_ERA5_1960_.csv', index=False)


In [66]:
LEE_dat_14 = LEE_detection(dat_14, n_dat_14, dur = 14)

In [67]:
LEE_dat_14

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-12-06,1960-12-19,1960-12-06 00:00:00.000000007,339,352,346,14,1462.470364,1319.908104,140.107165,18478.713458,1,ERA5_week,ERA5_hist_week,1960
1,1960-12-07,1960-12-20,1960-12-07 00:00:00.000000006,340,353,346,14,1462.470364,1326.178826,141.769729,18566.503568,2,ERA5_week,ERA5_hist_week,1960
2,1960-12-08,1960-12-21,1960-12-08 00:00:00.000000005,341,354,346,14,1462.470364,1328.602852,143.212437,18600.439932,3,ERA5_week,ERA5_hist_week,1960
3,1960-12-11,1960-12-24,1960-12-11 00:00:00.000000002,344,357,346,14,1462.470364,1330.305104,101.957148,18624.271458,4,ERA5_week,ERA5_hist_week,1960
4,1961-12-13,1961-12-26,1961-12-13 00:00:00.000000013,711,724,724,14,1540.029319,1333.213482,162.683705,18664.988745,5,ERA5_week,ERA5_hist_week,1961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,2022-12-01,2022-12-14,2022-12-01 00:00:00.000000009,22964,22977,22973,14,1433.392018,1335.129181,100.798361,18691.808535,161,ERA5_week,ERA5_hist_week,2022
161,2022-12-02,2022-12-15,2022-12-02 00:00:00.000000008,22965,22978,22973,14,1433.392018,1336.000011,101.372024,18704.000154,162,ERA5_week,ERA5_hist_week,2022
162,2022-12-03,2022-12-16,2022-12-03 00:00:00.000000013,22966,22979,22979,14,1491.687275,1353.979124,105.003797,18955.707739,163,ERA5_week,ERA5_hist_week,2022
163,2022-12-04,2022-12-17,2022-12-04 00:00:00.000000012,22967,22980,22979,14,1491.687275,1359.222781,99.436052,19029.118939,164,ERA5_week,ERA5_hist_week,2022


In [68]:
# If the date_start is the the next day from the line before and they are the same ESM_run they should get the same event number
new_run = LEE_dat_14['ESM_run'].ne(LEE_dat_14['ESM_run'].shift())
not_consecutive = (LEE_dat_14['date_start'] - LEE_dat_14['date_start'].shift()) != pd.Timedelta(days=1)

# Combine them — whenever either is True, that row is the start of a new event
is_new_event = new_run | not_consecutive

# 4. Cum-sum that to get a 1-based event ID
LEE_dat_14['event'] = is_new_event.cumsum()
# LEE_dat_14

In [69]:
LEE_dat_14.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_14_ERA5_1960_.csv', index=False)
# LEE_dat_14.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_14.csv', index=False)

LEE_dat_14

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-12-06,1960-12-19,1960-12-06 00:00:00.000000007,339,352,346,14,1462.470364,1319.908104,140.107165,18478.713458,1,ERA5_week,ERA5_hist_week,1960
1,1960-12-07,1960-12-20,1960-12-07 00:00:00.000000006,340,353,346,14,1462.470364,1326.178826,141.769729,18566.503568,1,ERA5_week,ERA5_hist_week,1960
2,1960-12-08,1960-12-21,1960-12-08 00:00:00.000000005,341,354,346,14,1462.470364,1328.602852,143.212437,18600.439932,1,ERA5_week,ERA5_hist_week,1960
3,1960-12-11,1960-12-24,1960-12-11 00:00:00.000000002,344,357,346,14,1462.470364,1330.305104,101.957148,18624.271458,2,ERA5_week,ERA5_hist_week,1960
4,1961-12-13,1961-12-26,1961-12-13 00:00:00.000000013,711,724,724,14,1540.029319,1333.213482,162.683705,18664.988745,3,ERA5_week,ERA5_hist_week,1961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,2022-12-01,2022-12-14,2022-12-01 00:00:00.000000009,22964,22977,22973,14,1433.392018,1335.129181,100.798361,18691.808535,38,ERA5_week,ERA5_hist_week,2022
161,2022-12-02,2022-12-15,2022-12-02 00:00:00.000000008,22965,22978,22973,14,1433.392018,1336.000011,101.372024,18704.000154,38,ERA5_week,ERA5_hist_week,2022
162,2022-12-03,2022-12-16,2022-12-03 00:00:00.000000013,22966,22979,22979,14,1491.687275,1353.979124,105.003797,18955.707739,38,ERA5_week,ERA5_hist_week,2022
163,2022-12-04,2022-12-17,2022-12-04 00:00:00.000000012,22967,22980,22979,14,1491.687275,1359.222781,99.436052,19029.118939,38,ERA5_week,ERA5_hist_week,2022


In [70]:
# find the index of the row with the highest RL_cumulative in each event
idx = LEE_dat_14.groupby('event')['RL_cumulative'].idxmax()

# select only those rows
LEE_dat_14_max_cum_RL = LEE_dat_14.loc[idx].reset_index(drop=True)
LEE_dat_14_max_cum_RL

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-12-08,1960-12-21,1960-12-08 00:00:00.000000005,341,354,346,14,1462.470364,1328.602852,143.212437,18600.439932,1,ERA5_week,ERA5_hist_week,1960
1,1960-12-11,1960-12-24,1960-12-11 00:00:00.000000002,344,357,346,14,1462.470364,1330.305104,101.957148,18624.271458,2,ERA5_week,ERA5_hist_week,1960
2,1961-12-15,1961-12-28,1961-12-15 00:00:00.000000011,713,726,724,14,1540.029319,1395.921231,96.892106,19542.897236,3,ERA5_week,ERA5_hist_week,1961
3,1962-12-17,1962-12-30,1962-12-17 00:00:00.000000012,1080,1093,1092,14,1515.289946,1331.505814,148.778980,18641.081397,4,ERA5_week,ERA5_hist_week,1962
4,1963-11-28,1963-12-11,1963-11-28 00:00:00.000000012,1426,1439,1438,14,1477.355117,1342.784636,105.705116,18798.984901,5,ERA5_week,ERA5_hist_week,1963
5,1963-12-05,1963-12-18,1963-12-05 00:00:00.000000010,1433,1446,1443,14,1506.276972,1385.417392,166.705017,19395.843482,6,ERA5_week,ERA5_hist_week,1963
6,1964-01-07,1964-01-20,1964-01-07 00:00:00.000000005,1466,1479,1471,14,1476.483679,1381.043362,66.114489,19334.607071,7,ERA5_week,ERA5_hist_week,1964
7,1964-12-16,1964-12-29,1964-12-16 00:00:00.000000011,1809,1822,1820,14,1477.550480,1378.353728,77.178322,19296.952198,8,ERA5_week,ERA5_hist_week,1964
8,1965-01-19,1965-02-01,1965-01-19 00:00:00.000000011,1843,1856,1854,14,1420.199251,1320.267749,94.308858,18483.748484,9,ERA5_week,ERA5_hist_week,1965
9,1968-11-30,1968-12-13,1968-11-30 00:00:00.000000012,3253,3266,3265,14,1489.607944,1321.226886,153.892149,18497.176405,10,ERA5_week,ERA5_hist_week,1968


In [71]:
# LEE_dat_14_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_dat_14_selection.csv', index=False)
LEE_dat_14_max_cum_RL.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_dat_14_selection_ERA5_1960_.csv', index=False)


### Event detection for variable length

In [74]:
def LEE_detection_vl(events, n_events, t=RL['time'].values, minDuration=1):
    LEE_records = []

    for ev in range(1, n_events + 1):
        event_duration = (events == ev).sum()
        if event_duration < minDuration:
            continue

        start_idx = np.where(events == ev)[0][0]
        end_idx = np.where(events == ev)[0][-1]

        date_start = t[start_idx]
        date_end = t[end_idx]

        LEE_start = np.where(t == date_start)[0][0]
        LEE_end = np.where(t == date_end)[0][0]

        RL_run = RL.values
        RL_LEE = RL_run[LEE_start:LEE_end + 1]
        LEE_peak = np.argmax(RL_LEE)

        record = {
            'date_start': date_start,
            'date_end': date_end,
            'date_peak': date_start + LEE_peak,
            # 'date_start_old': RL['old_time'][i][LEE_start].values,
            # 'date_end_old': RL['old_time'][i][LEE_end].values,
            # 'date_peak_old': RL['old_time'][i][LEE_start + LEE_peak].values,
            'index_start': LEE_start,
            'index_end': LEE_end,
            'index_peak': LEE_start + LEE_peak,
            'duration': len(RL_LEE),
            'RL_max': RL_LEE[LEE_peak],
            'RL_mean': RL_LEE.mean(),
            'RL_var': np.sqrt(RL_LEE.var()),
            'RL_cumulative': RL_LEE.sum(),
            'event': ev,
            'ESM': str(RL.ESM.values),
            'ESM_run': str(RL.ESM_run.values),
            'year': RL.time.dt.year[start_idx].values,
            # 'doy' : RL.doy[start_idx].values#,
            # 'winter': RL.winter_year[start_idx].values,
            # 'day_of_winter': RL.day_of_winter[start_idx].values
        }

        # # Optional additional metrics if available
        # # (you could generalize this to loop over var names too)
        # for var in ['prod', 'demand', 'pot']:
        #     try:
        #         var_data = RL[var].sel(ESM_run=RL.ESM_run[i]).values[LEE_start:LEE_end + 1]
        #         record[f'{var}_max'] = var_data.max()
        #         record[f'{var}_mean'] = var_data.mean()
        #         record[f'{var}_var'] = np.sqrt(var_data.var())
        #         record[f'{var}_cumulative'] = var_data.sum()
        #     except KeyError:
        #         pass  # Variable doesn't exist, skip

        LEE_records.append(record)

    return pd.DataFrame(LEE_records)

In [75]:
LEE_vl = LEE_detection_vl(events_vl, n_events_vl)

In [76]:
# LEE_vl.to_csv('/climca/people/onennecke/model_output/LEE_detection/not_bc_adj/LEE_vl.csv', index=False)
LEE_vl.to_csv('/climca/people/onennecke/model_output/LEE_detection/LEE_vl_ERA5_1960_.csv', index=False)


In [77]:
LEE_vl

,date_start,date_end,date_peak,index_start,index_end,index_peak,duration,RL_max,RL_mean,RL_var,RL_cumulative,event,ESM,ESM_run,year
0,1960-01-03,1960-01-04,1960-01-03 00:00:00.000000000,2,3,2,2,1380.195711,1355.541575,24.654136,2711.083150,1,ERA5_week,ERA5_hist_week,1960
1,1960-01-10,1960-01-10,1960-01-10 00:00:00.000000000,9,9,9,1,1401.099958,1401.099958,0.000000,1401.099958,2,ERA5_week,ERA5_hist_week,1960
2,1960-01-12,1960-01-12,1960-01-12 00:00:00.000000000,11,11,11,1,1392.550256,1392.550256,0.000000,1392.550256,3,ERA5_week,ERA5_hist_week,1960
3,1960-01-15,1960-01-17,1960-01-15 00:00:00.000000002,14,16,16,3,1481.905678,1471.177659,9.884807,4413.532978,4,ERA5_week,ERA5_hist_week,1960
4,1960-01-21,1960-01-21,1960-01-21 00:00:00.000000000,20,20,20,1,1410.415754,1410.415754,0.000000,1410.415754,5,ERA5_week,ERA5_hist_week,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,2024-11-29,2024-11-29,2024-11-29 00:00:00.000000000,23692,23692,23692,1,1365.633236,1365.633236,0.000000,1365.633236,775,ERA5_week,ERA5_hist_week,2024
775,2024-12-04,2024-12-04,2024-12-04 00:00:00.000000000,23697,23697,23697,1,1343.232708,1343.232708,0.000000,1343.232708,776,ERA5_week,ERA5_hist_week,2024
776,2024-12-11,2024-12-13,2024-12-11 00:00:00.000000000,23704,23706,23704,3,1434.225659,1423.271436,14.075485,4269.814307,777,ERA5_week,ERA5_hist_week,2024
777,2024-12-24,2024-12-24,2024-12-24 00:00:00.000000000,23717,23717,23717,1,1337.232681,1337.232681,0.000000,1337.232681,778,ERA5_week,ERA5_hist_week,2024
